In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from ipywidgets import (
    IntSlider, FloatSlider, HTML, HTMLMath,
    VBox, HBox, Layout
)
from IPython.display import display

plt.ioff()

# ------------------------------------------------------------
# Jupyter / Binder display settings
# ------------------------------------------------------------

display(HTML("""
<style>
.container {
    width: 98% !important;
    max-width: none !important;
}

.output_area,
.output_subarea {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
    box-shadow: none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow: visible !important;
    resize: none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display: none !important;
}

.cheb-title {
    font-family: Arial, sans-serif;
    font-size: 20px;
    font-weight: bold;
    color: #6f3fa0;
}

.cheb-label {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
}

.cheb-value {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
    color: #0b3d91;
}
</style>
"""))

# ------------------------------------------------------------
# Symbolic variable
# ------------------------------------------------------------

x = sp.symbols('x', real=True)

# ------------------------------------------------------------
# Documentation
# ------------------------------------------------------------

documentation = HTML("""
<div style="
    width:930px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="cheb-title" style="margin-bottom:8px;">
Chebyshev Polynomials of the First Kind
</div>

<div style="margin-bottom:5px;">
The Chebyshev polynomials of the first kind are generated recursively
from T₀(x)=1 and T₁(x)=x using
Tₙ₊₁(x)=2xTₙ(x)−Tₙ₋₁(x).
</div>

<div style="margin-bottom:5px;">
For −1 ≤ x ≤ 1 they satisfy
Tₙ(x)=cos(n arccos x) and oscillate between −1 and +1.
</div>

<div>
<b>This notebook:</b> constructs Tₙ(x) symbolically from the recurrence,
plots it on [-1,1], and simultaneously displays the function
1+ε²Tₙ²(x).
</div>

</div>
""")

# ------------------------------------------------------------
# Symbolic construction from the recurrence
# ------------------------------------------------------------

def construct_chebyshev(N):
    if N == 0:
        return sp.Integer(1)

    if N == 1:
        return x

    T_previous = sp.Integer(1)
    T_current = x

    for k in range(1, N):
        T_next = sp.expand(2*x*T_current - T_previous)
        T_previous = T_current
        T_current = T_next

    return sp.expand(T_current)

# ------------------------------------------------------------
# Slider styles
# ------------------------------------------------------------

slider_style = {'description_width': '0px'}

slider_layout = Layout(width='190px')
label_layout = Layout(width='100px', min_width='100px')
value_layout = Layout(
    width='60px',
    min_width='60px',
    margin='0px 0px 0px 6px'
)

row_layout = Layout(
    width='380px',
    height='38px',
    align_items='center'
)

# ------------------------------------------------------------
# Order N
# ------------------------------------------------------------

N_slider = IntSlider(
    min=0,
    max=10,
    step=1,
    value=5,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

N_label = HTML(
    '<div class="cheb-label">Order N:</div>',
    layout=label_layout
)

N_value = HTML(
    '<div class="cheb-value">5</div>',
    layout=value_layout
)

N_row = HBox(
    [N_label, N_slider, N_value],
    layout=row_layout
)

# ------------------------------------------------------------
# Epsilon
# ------------------------------------------------------------

epsilon_slider = FloatSlider(
    min=0.05,
    max=1.00,
    step=0.05,
    value=0.50,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

epsilon_label = HTML(
    '<div class="cheb-label">Parameter ε:</div>',
    layout=label_layout
)

epsilon_value = HTML(
    '<div class="cheb-value">0.50</div>',
    layout=value_layout
)

epsilon_row = HBox(
    [epsilon_label, epsilon_slider, epsilon_value],
    layout=row_layout
)

# ------------------------------------------------------------
# Parameters panel
# ------------------------------------------------------------

parameters_panel = VBox(
    [
        HTML("""
        <div class="cheb-title" style="margin-bottom:9px;">
            Parameters
        </div>
        """),
        N_row,
        epsilon_row
    ],
    layout=Layout(
        width='400px',
        padding='10px 12px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ------------------------------------------------------------
# Symbolic result
# ------------------------------------------------------------

initial_math = HTMLMath(
    value=r'\(T_0(x)=1,\;T_1(x)=x\)'
)

recurrence_math = HTMLMath(
    value=r'\(T_{N+1}(x)=2xT_N(x)-T_{N-1}(x)\)'
)

polynomial_math = HTMLMath()

T_zero_math = HTMLMath()
T_one_math = HTMLMath()
F_zero_math = HTMLMath()

# The expressions are separate widgets.
# Therefore NO \quad or \qquad is used.

property_row = HBox(
    [
        T_zero_math,
        T_one_math,
        F_zero_math
    ],
    layout=Layout(
        width='500px',
        gap='18px',
        align_items='center',
        overflow='visible'
    )
)

T_zero_math.layout = Layout(width='105px')
T_one_math.layout = Layout(width='105px')
F_zero_math.layout = Layout(width='240px')

# ------------------------------------------------------------
# Symbolic panel
# ------------------------------------------------------------

symbolic_panel = VBox(
    [
        HTML("""
        <div class="cheb-title" style="margin-bottom:9px;">
            Symbolic Result
        </div>
        """),
        initial_math,
        recurrence_math,
        polynomial_math,
        property_row
    ],
    layout=Layout(
        width='510px',
        padding='10px 12px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ------------------------------------------------------------
# Top row
# ------------------------------------------------------------

top_row = HBox(
    [
        parameters_panel,
        symbolic_panel
    ],
    layout=Layout(
        width='930px',
        gap='12px',
        align_items='stretch',
        overflow='visible'
    )
)

# ------------------------------------------------------------
# Numerical grid
# ------------------------------------------------------------

x_values = np.linspace(-1.0, 1.0, 1600)

T_initial_symbolic = construct_chebyshev(N_slider.value)

T_initial_function = sp.lambdify(
    x,
    T_initial_symbolic,
    modules='numpy'
)

T_initial_values = np.asarray(
    T_initial_function(x_values),
    dtype=float
)

if T_initial_values.ndim == 0:
    T_initial_values = np.full(
        len(x_values),
        float(T_initial_values)
    )

epsilon_initial = epsilon_slider.value

F_initial_values = (
    1.0
    + epsilon_initial**2 * T_initial_values**2
)

# ------------------------------------------------------------
# Figure 1: T_N(x)
# ------------------------------------------------------------

fig_T, ax_T = plt.subplots(figsize=(4.3, 3.8))

fig_T.canvas.header_visible = False
fig_T.canvas.footer_visible = False
fig_T.canvas.toolbar_visible = False

fig_T.canvas.layout = Layout(
    width='430px',
    height='380px',
    overflow='visible'
)

ax_T.set_title(
    'Chebyshev Polynomial Tₙ(x)',
    fontsize=13,
    fontweight='bold',
    color='#6f3fa0'
)

ax_T.set_xlabel('x', fontsize=10)
ax_T.set_ylabel('Tₙ(x)', fontsize=10)

ax_T.set_xlim(-1.0, 1.0)
ax_T.set_ylim(-1.15, 1.15)

ax_T.axhline(0.0, linewidth=1.0)

ax_T.grid(
    True,
    linestyle=':',
    alpha=0.40
)

line_T, = ax_T.plot(
    x_values,
    T_initial_values,
    linewidth=2.0
)

ax_T.axhline(
    1.0,
    linestyle='--',
    linewidth=1.0,
    alpha=0.60
)

ax_T.axhline(
    -1.0,
    linestyle='--',
    linewidth=1.0,
    alpha=0.60
)

fig_T.subplots_adjust(
    left=0.14,
    right=0.97,
    top=0.89,
    bottom=0.14
)

# ------------------------------------------------------------
# Figure 2: 1 + epsilon² T_N²(x)
# ------------------------------------------------------------

fig_F, ax_F = plt.subplots(figsize=(4.3, 3.8))

fig_F.canvas.header_visible = False
fig_F.canvas.footer_visible = False
fig_F.canvas.toolbar_visible = False

fig_F.canvas.layout = Layout(
    width='430px',
    height='380px',
    overflow='visible'
)

ax_F.set_title(
    'Function 1 + ε²Tₙ²(x)',
    fontsize=13,
    fontweight='bold',
    color='#0b3d91'
)

ax_F.set_xlabel('x', fontsize=10)
ax_F.set_ylabel('1 + ε²Tₙ²(x)', fontsize=10)

ax_F.set_xlim(-1.0, 1.0)
ax_F.set_ylim(0.95, 2.05)

ax_F.axhline(1.0, linewidth=1.0)

ax_F.grid(
    True,
    linestyle=':',
    alpha=0.40
)

line_F, = ax_F.plot(
    x_values,
    F_initial_values,
    linewidth=2.0
)

fig_F.subplots_adjust(
    left=0.15,
    right=0.97,
    top=0.89,
    bottom=0.14
)

# ------------------------------------------------------------
# Figures row
# ------------------------------------------------------------

figures_row = HBox(
    [
        fig_T.canvas,
        fig_F.canvas
    ],
    layout=Layout(
        width='875px',
        gap='12px',
        align_items='flex-start',
        overflow='visible'
    )
)

# ------------------------------------------------------------
# Update
# ------------------------------------------------------------

def update_notebook(change=None):

    N = N_slider.value
    epsilon = epsilon_slider.value

    N_value.value = (
        f'<div class="cheb-value">{N}</div>'
    )

    epsilon_value.value = (
        f'<div class="cheb-value">{epsilon:.2f}</div>'
    )

    # Symbolic construction
    T_symbolic = construct_chebyshev(N)

    # Numerical evaluation
    T_function = sp.lambdify(
        x,
        T_symbolic,
        modules='numpy'
    )

    T_values = np.asarray(
        T_function(x_values),
        dtype=float
    )

    if T_values.ndim == 0:
        T_values = np.full(
            len(x_values),
            float(T_values)
        )

    F_values = (
        1.0
        + epsilon**2 * T_values**2
    )

    # Update curves only
    line_T.set_ydata(T_values)
    line_F.set_ydata(F_values)

    # Symbolic polynomial
    polynomial_math.value = (
        r'\(T_{'
        + str(N)
        + r'}(x)='
        + sp.latex(T_symbolic)
        + r'\)'
    )

    # Symbolic special values
    T_at_zero = sp.simplify(
        T_symbolic.subs(x, 0)
    )

    T_at_one = sp.simplify(
        T_symbolic.subs(x, 1)
    )

    F_at_zero = (
        1.0
        + epsilon**2 * float(T_at_zero)**2
    )

    # --------------------------------------------------------
    # NO \quad
    # NO \qquad
    # --------------------------------------------------------

    T_zero_math.value = (
        r'\(T_N(0)='
        + sp.latex(T_at_zero)
        + r'\)'
    )

    T_one_math.value = (
        r'\(T_N(1)='
        + sp.latex(T_at_one)
        + r'\)'
    )

    F_zero_math.value = (
        r'\(1+\varepsilon^2T_N^2(0)='
        + f'{F_at_zero:.4f}'
        + r'\)'
    )

    fig_T.canvas.draw_idle()
    fig_F.canvas.draw_idle()

# ------------------------------------------------------------
# Slider callbacks
# ------------------------------------------------------------

N_slider.observe(
    update_notebook,
    names='value'
)

epsilon_slider.observe(
    update_notebook,
    names='value'
)

# Initial update
update_notebook()

# ------------------------------------------------------------
# Complete notebook
# ------------------------------------------------------------

main_layout = VBox(
    [
        documentation,
        top_row,
        figures_row
    ],
    layout=Layout(
        width='950px',
        gap='10px',
        overflow='visible'
    )
)

display(main_layout)